<a href="https://colab.research.google.com/github/Abhishek2993/BrainCancer-Detection-MRI-DATA-/blob/main/Project_BCDCipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCDC: Brain Cancer Detection & Classification from MRI

This notebook trains and evaluates a **3-class brain-tumour MRI classifier** (glioma / meningioma / `brain_tumor`) with a MobileNetV2 transfer-learning model.

What it does:

1. Downloads the Kaggle *Brain Cancer MRI* dataset and **audits** it (class counts, image modes, exact and near-duplicate images).
2. Builds a **train / validation / test split grouped by near-duplicate cluster**, so copies of the same image can't end up on both sides of the split.
3. Trains MobileNetV2 in **two stages** (frozen backbone, then fine-tuning the top blocks) with early stopping, over **several random seeds**.
4. Reports **held-out test** metrics as mean ± std over seeds: accuracy, balanced accuracy, macro-F1, per-class sensitivity / specificity / precision, one-vs-rest AUROC, a confusion matrix, and expected calibration error (ECE).
5. Calibrates the confidence scores with **temperature scaling** (fitted on the validation set only).
6. Provides a single-image **inference** function that handles any image mode, applies softmax once, and flags low-confidence or non-MRI-looking inputs.

> ⚠️ **Research and educational use only.** This model is not a medical device and must not be used for diagnosis. The dataset has **no "no tumour" class**, so the model assumes a tumour is present and only guesses which type.

**Runtime:** use a GPU runtime in Colab (*Runtime → Change runtime type → T4 GPU*). A full run with 3 seeds takes roughly 30–60 minutes.

## 1. Setup and configuration

In [ ]:
import hashlib
import io
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, f1_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---- Configuration -------------------------------------------------------
SPLIT_SEED = 42              # fixes the train/val/test split
TRAIN_SEEDS = [0, 1, 2]      # one model per seed; results reported as mean ± std (use 5 for final numbers)
VAL_FRAC, TEST_FRAC = 0.15, 0.15

IMG_SIZE = (224, 224)        # MobileNetV2's native input resolution
BATCH_SIZE = 32

HEAD_EPOCHS, HEAD_LR = 15, 1e-3             # stage 1: frozen backbone, train the new head
FINETUNE_EPOCHS, FINETUNE_LR = 20, 1e-5     # stage 2: unfreeze the top of the backbone
FINETUNE_FROM_LAYER = 100                   # MobileNetV2 has 154 layers; layers >= 100 are unfrozen
PATIENCE = 4                                # early-stopping patience (epochs, on val loss)

DUP_HAMMING = 4              # two images whose 64-bit dHashes differ in <= 4 bits count as near-duplicates
MIN_CONFIDENCE = 0.70        # below this calibrated probability, inference reports "uncertain"
BACKBONE_WEIGHTS = "imagenet"

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"

# Optional overrides, used for quick local smoke tests. Leave unset in Colab.
DATASET_OVERRIDE = os.environ.get("BCDC_DATASET_PATH")
if os.environ.get("BCDC_QUICK_RUN") == "1":
    TRAIN_SEEDS, HEAD_EPOCHS, FINETUNE_EPOCHS, IMG_SIZE = [0, 1], 1, 1, (96, 96)
if os.environ.get("BCDC_WEIGHTS") == "none":
    BACKBONE_WEIGHTS = None

FIG_DIR.mkdir(parents=True, exist_ok=True)
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# Readable names for the dataset's folder names. The source dataset labels its third
# class only as "brain_tumor" and does not say which tumour type it is, so we keep
# that label as-is rather than guessing (it is NOT necessarily pituitary).
DISPLAY_NAMES = {
    "brain_glioma": "Glioma",
    "brain_menin": "Meningioma",
    "brain_tumor": "Brain tumour (unspecified type)",
}

## 2. Download the dataset

In [ ]:
def find_class_root(root):
    # Returns the first directory under `root` whose sub-folders all contain images.
    for dirpath, dirnames, _ in os.walk(root):
        subs = sorted(d for d in dirnames if not d.startswith("."))
        if len(subs) >= 2 and all(
            any(f.lower().endswith(IMG_EXTS) for f in os.listdir(os.path.join(dirpath, d)))
            for d in subs
        ):
            return Path(dirpath)
    raise FileNotFoundError(f"No folder of class sub-folders found under {root}")


if DATASET_OVERRIDE:
    DATASET_PATH = find_class_root(DATASET_OVERRIDE)
else:
    import kagglehub
    path = kagglehub.dataset_download("orvile/brain-cancer-mri-dataset")
    expected = Path(path) / "Brain_Cancer raw MRI data" / "Brain_Cancer"
    DATASET_PATH = expected if expected.is_dir() else find_class_root(path)

CLASS_NAMES = sorted(d.name for d in DATASET_PATH.iterdir() if d.is_dir() and not d.name.startswith("."))
NUM_CLASSES = len(CLASS_NAMES)
print("Dataset:", DATASET_PATH)
print("Classes:", CLASS_NAMES)

## 3. Data audit

Before splitting, we check what's actually in the data:

* **class balance**, **image modes** (RGB / grayscale / RGBA) and **sizes**;
* **exact duplicates** (identical file bytes);
* **near-duplicates**: we compute a 64-bit difference hash (dHash) of each image *and of its 8 flipped/rotated versions*. Two images are near-duplicates when any pair of hashes differs in at most `DUP_HAMMING` bits. This catches resaved copies and flip/90°-rotation augmentations.

Near-duplicates are merged into **groups**, and whole groups are assigned to one split. This stops copies of an image from appearing in both training and test, which would inflate the score.

**Limitation:** the dataset has no patient IDs, so *different* slices from the same patient can still land in different splits. Test scores may therefore still be somewhat optimistic.

In [ ]:
def dhash_bits(img_l, size=8):
    # 64-bit difference hash of a grayscale PIL image, as a Python int.
    small = np.asarray(img_l.resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return int(np.packbits(bits).view(">u8")[0])


ROTATIONS = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270]


def dihedral_hashes(img_l):
    # dHashes of the image under all 8 flips/90-degree rotations (index 0 = original).
    small = img_l.resize((64, 64), Image.BILINEAR)   # downscale once; flips/rotations are exact on the small copy
    out = []
    for base in (small, small.transpose(Image.Transpose.FLIP_LEFT_RIGHT)):
        for rot in ROTATIONS:
            out.append(dhash_bits(base if rot is None else base.transpose(rot)))
    return out


records = []
for label, cls in enumerate(CLASS_NAMES):
    for f in sorted((DATASET_PATH / cls).iterdir()):
        if not f.name.lower().endswith(IMG_EXTS):
            continue
        raw = f.read_bytes()
        try:
            with Image.open(io.BytesIO(raw)) as im:
                mode, size = im.mode, im.size
                hashes = dihedral_hashes(im.convert("L"))
        except Exception as e:  # unreadable file
            print("Skipping unreadable file:", f, e)
            continue
        records.append(dict(path=str(f), cls=cls, label=label, mode=mode,
                            width=size[0], height=size[1],
                            md5=hashlib.md5(raw).hexdigest(), hashes=hashes))

df = pd.DataFrame(records)
print(f"{len(df)} readable images\n")
print(df["cls"].value_counts().rename("images per class").to_string(), "\n")
print(df["mode"].value_counts().rename("image modes").to_string(), "\n")
print(df.groupby(["width", "height"]).size().sort_values(ascending=False).head(5)
        .rename("most common sizes").to_string())

In [ ]:
_BYTE_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def popcount64(x):
    # Number of set bits in each uint64 (works on any NumPy version).
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return _BYTE_POPCOUNT[x.view(np.uint8)].reshape(x.shape + (8,)).sum(axis=-1)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, i, j):
        ri, rj = self.find(i), self.find(j)
        if ri != rj:
            self.parent[rj] = ri


n = len(df)
uf = UnionFind(n)

# Exact duplicates (same bytes).
for _, idx in df.groupby("md5").indices.items():
    for j in idx[1:]:
        uf.union(idx[0], j)
n_exact = int(df.duplicated("md5").sum())

# Near-duplicates: compare each image's original hash against every other image's 8 dihedral hashes.
orig = np.array([h[0] for h in df["hashes"]], dtype=np.uint64)
allh = np.array(df["hashes"].tolist(), dtype=np.uint64)          # shape (n, 8)
CHUNK = 64                                                         # rows per batch, keeps memory small
near_pairs = set()
for start in range(0, n, CHUNK):
    a = orig[start:start + CHUNK]                                  # (c,)
    dist = popcount64(np.bitwise_xor(a[:, None, None], allh[None, :, :])).min(axis=2)  # (c, n)
    for r, j in zip(*np.nonzero(dist <= DUP_HAMMING)):
        i = start + int(r)
        if i != j:
            uf.union(i, int(j))
            near_pairs.add((min(i, int(j)), max(i, int(j))))
near_pairs = len(near_pairs)

df["group"] = [uf.find(i) for i in range(n)]
group_sizes = df.groupby("group").size()
group_classes = df.groupby("group")["label"].nunique()
conflict_groups = set(group_classes[group_classes > 1].index)

print(f"Exact duplicate files:                 {n_exact}")
print(f"Near-duplicate pairs (<= {DUP_HAMMING} bits):     {near_pairs}")
print(f"Groups:                                {len(group_sizes)} (from {n} images)")
print(f"Images in groups of size > 1:          {int(group_sizes[group_sizes > 1].sum())}")
print(f"Groups spanning more than one class:   {len(conflict_groups)} "
      f"({int(df['group'].isin(conflict_groups).sum())} images)")

In [ ]:
# Show a few near-duplicate groups so they can be checked by eye.
multi = group_sizes[group_sizes > 1].sort_values(ascending=False).index[:4]
if len(multi):
    fig, axes = plt.subplots(len(multi), 4, figsize=(10, 2.6 * len(multi)), squeeze=False)
    for row, g in zip(axes, multi):
        members = df[df["group"] == g].head(4)
        for ax in row:
            ax.axis("off")
        for ax, (_, r) in zip(row, members.iterrows()):
            ax.imshow(Image.open(r["path"]).convert("L"), cmap="gray")
            ax.set_title(f"{r['cls']}\n{Path(r['path']).name[:22]}", fontsize=7)
    plt.suptitle("Examples of near-duplicate groups (each row = one group)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "near_duplicate_examples.png", dpi=120)
    plt.show()
else:
    print("No near-duplicate groups found.")

## 4. Grouped, stratified train / validation / test split

Groups whose images carry **different class labels** (the same picture filed under two classes) are dropped, because we can't tell which label is right. The remaining groups are split ~70 / 15 / 15, stratified by class. The split is saved to `outputs/split.csv`.

In [ ]:
clean = df[~df["group"].isin(conflict_groups)].copy()
groups = clean.groupby("group")["label"].first()

g_trainval, g_test = train_test_split(groups.index, test_size=TEST_FRAC,
                                      stratify=groups.values, random_state=SPLIT_SEED)
g_train, g_val = train_test_split(g_trainval, test_size=VAL_FRAC / (1 - TEST_FRAC),
                                  stratify=groups.loc[g_trainval].values, random_state=SPLIT_SEED)
split_of = {**{g: "train" for g in g_train}, **{g: "val" for g in g_val}, **{g: "test" for g in g_test}}
clean["split"] = clean["group"].map(split_of)

# Sanity check: no group appears in two splits.
assert clean.groupby("group")["split"].nunique().max() == 1

clean[["path", "cls", "label", "group", "split"]].assign(
    path=lambda d: d["path"].map(lambda p: str(Path(p).relative_to(DATASET_PATH)))
).to_csv(OUT_DIR / "split.csv", index=False)

print(pd.crosstab(clean["cls"], clean["split"], margins=True)[["train", "val", "test", "All"]])

## 5. Input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)  # any mode -> RGB
    img = tf.image.resize(img, IMG_SIZE)                                                  # float32, 0-255
    return tf.saturate_cast(tf.round(img), tf.uint8), label   # uint8 keeps the in-memory cache small


def make_ds(split, shuffle=False, seed=0):
    part = clean[clean["split"] == split]
    ds = tf.data.Dataset.from_tensor_slices((part["path"].values, part["label"].values.astype("int32")))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    return ds.map(lambda x, y: (tf.cast(x, tf.float32), y)).batch(BATCH_SIZE).prefetch(AUTOTUNE)


val_ds, test_ds = make_ds("val"), make_ds("test")
y_val = clean.loc[clean["split"] == "val", "label"].values
y_test = clean.loc[clean["split"] == "test", "label"].values

# Visual check of a few training images
preview = make_ds("train", shuffle=True)
plt.figure(figsize=(7, 7))
for images, labels in preview.take(1):
    for i in range(min(9, len(labels))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(DISPLAY_NAMES.get(CLASS_NAMES[labels[i]], CLASS_NAMES[labels[i]]), fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Model

* Input: raw RGB pixels in **[0, 255]**. Preprocessing is built into the model, so training and inference can't drift apart.
* **Preprocessing:** MobileNetV2 was pretrained on inputs scaled to **[-1, 1]** (`x / 127.5 - 1`). The old notebook used `Rescaling(1/255)`, which gives [0, 1] and doesn't match the pretrained weights.
* Augmentation (only active during training): horizontal flip, small rotation, zoom and contrast changes.
* The backbone always runs with `training=False`, so its BatchNorm statistics stay fixed during fine-tuning.
* The model ends in **one** softmax. Its output is already a probability distribution.

In [ ]:
def build_model(seed):
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)   # seeds Python, NumPy and TensorFlow

    augment = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(0.05, fill_mode="constant", seed=seed),
        layers.RandomZoom(0.1, fill_mode="constant", seed=seed),
        layers.RandomContrast(0.1, seed=seed),
    ], name="augmentation")

    base = keras.applications.MobileNetV2(include_top=False, weights=BACKBONE_WEIGHTS,
                                          input_shape=IMG_SIZE + (3,))
    base.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,), name="image_rgb_0_255")
    x = augment(inputs)
    x = layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_v2_preprocess")(x)  # -> [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3, seed=seed)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="probs")(x)
    return keras.Model(inputs, outputs, name="bcdc_mobilenetv2"), base


def early_stop():
    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                         restore_best_weights=True)


def train_one_seed(seed):
    model, base = build_model(seed)
    train_ds = make_ds("train", shuffle=True, seed=seed)

    # Stage 1: train only the new classification head.
    model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
                   callbacks=[early_stop()], verbose=2)

    # Stage 2: unfreeze the top of the backbone (BatchNorm layers stay frozen) with a low LR.
    base.trainable = True
    for i, layer in enumerate(base.layers):
        layer.trainable = i >= FINETUNE_FROM_LAYER and not isinstance(layer, layers.BatchNormalization)
    model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
                   callbacks=[early_stop()], verbose=2)

    history = {k: h1.history[k] + h2.history[k] for k in h1.history}
    history["stage1_epochs"] = len(h1.history["loss"])
    return model, history

## 7. Evaluation and calibration helpers

For each class *k* (treated one-vs-rest):

* **Sensitivity (recall):** the share of true class-*k* scans the model labels *k*. A missed glioma is the costliest error, so we report this per class rather than relying on overall accuracy.
* **Specificity:** the share of non-*k* scans that are *not* labelled *k*.
* **AUROC:** how well the class-*k* probability separates *k* from the rest, independent of any threshold.

**Expected calibration error (ECE)** measures whether "80% confident" really means right 80% of the time (15 equal-width bins). **Temperature scaling** divides the log-probabilities by a single number *T*, fitted to minimise validation negative log-likelihood. It changes how confident the model is, but never which class it predicts.

In [ ]:
def expected_calibration_error(y, probs, n_bins=15):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(ece)


def apply_temperature(probs, T):
    logp = np.log(np.clip(probs, 1e-12, 1.0)) / T   # log-probs are logits up to a constant
    logp -= logp.max(axis=-1, keepdims=True)
    e = np.exp(logp)
    return e / e.sum(axis=-1, keepdims=True)


def fit_temperature(y, probs):
    temps = np.exp(np.linspace(np.log(0.01), np.log(20.0), 600))
    nll = [-np.log(apply_temperature(probs, T)[np.arange(len(y)), y] + 1e-12).mean() for T in temps]
    return float(temps[int(np.argmin(nll))])


def evaluate(y, probs):
    pred = probs.argmax(axis=1)
    cm = confusion_matrix(y, pred, labels=range(NUM_CLASSES))
    total = cm.sum()
    out = {
        "accuracy": float((pred == y).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro")),
        "ece": expected_calibration_error(y, probs),
    }
    aurocs = []
    for k, cls in enumerate(CLASS_NAMES):
        tp = cm[k, k]
        fn = cm[k].sum() - tp
        fp = cm[:, k].sum() - tp
        tn = total - tp - fn - fp
        out[f"sensitivity[{cls}]"] = float(tp / max(tp + fn, 1))
        out[f"specificity[{cls}]"] = float(tn / max(tn + fp, 1))
        out[f"precision[{cls}]"] = float(tp / max(tp + fp, 1))
        yk = (y == k).astype(int)
        auc = roc_auc_score(yk, probs[:, k]) if 0 < yk.sum() < len(yk) else float("nan")
        out[f"auroc[{cls}]"] = float(auc)
        aurocs.append(auc)
    out["macro_auroc"] = float(np.nanmean(aurocs))
    return out, cm

## 8. Train and evaluate over several seeds

In [ ]:
runs = []
for seed in TRAIN_SEEDS:
    print(f"\n========== seed {seed} ==========")
    t0 = time.time()
    model, history = train_one_seed(seed)

    p_val = model.predict(val_ds, verbose=0)
    p_test = model.predict(test_ds, verbose=0)
    T = fit_temperature(y_val, p_val)                       # fitted on validation only
    test_raw, cm = evaluate(y_test, p_test)
    test_cal, _ = evaluate(y_test, apply_temperature(p_test, T))
    val_metrics, _ = evaluate(y_val, p_val)

    runs.append(dict(seed=seed, model=model, history=history, T=T, cm=cm,
                     p_test=p_test, p_val=p_val, test=test_raw, ece_calibrated=test_cal["ece"],
                     val_macro_f1=val_metrics["macro_f1"], val_loss=min(history["val_loss"]),
                     minutes=(time.time() - t0) / 60))
    print(f"seed {seed}: test acc={test_raw['accuracy']:.4f}  macro-F1={test_raw['macro_f1']:.4f}  "
          f"ECE {test_raw['ece']:.4f} -> {test_cal['ece']:.4f} (T={T:.2f})  [{runs[-1]['minutes']:.1f} min]")

In [ ]:
# ---- Results table: mean ± std over seeds (held-out TEST set) ----
per_seed = pd.DataFrame([{**r["test"], "ece_calibrated": r["ece_calibrated"], "temperature": r["T"]}
                         for r in runs], index=[f"seed {r['seed']}" for r in runs])
per_seed.to_csv(OUT_DIR / "test_metrics_per_seed.csv")

summary = pd.DataFrame({"mean": per_seed.mean(), "std": per_seed.std(ddof=1) if len(runs) > 1 else 0.0})
summary.to_csv(OUT_DIR / "test_metrics_summary.csv")


def fmt(metric, pct=True):
    m, s = summary.loc[metric, "mean"], summary.loc[metric, "std"]
    return f"{100 * m:.1f} ± {100 * s:.1f}" if pct else f"{m:.3f} ± {s:.3f}"


lines = [
    f"Test set: {len(y_test)} images · {len(runs)} seeds ({', '.join(str(r['seed']) for r in runs)}) · mean ± std",
    "",
    "| Metric | Value |", "|---|---|",
    f"| Accuracy (%) | {fmt('accuracy')} |",
    f"| Balanced accuracy (%) | {fmt('balanced_accuracy')} |",
    f"| Macro-F1 (%) | {fmt('macro_f1')} |",
    f"| Macro AUROC (one-vs-rest) | {fmt('macro_auroc', pct=False)} |",
    f"| ECE before → after temperature scaling | {fmt('ece', False)} → {fmt('ece_calibrated', False)} |",
    "",
    "| Class | Sensitivity (%) | Specificity (%) | Precision (%) | AUROC |", "|---|---|---|---|---|",
]
for cls in CLASS_NAMES:
    lines.append(f"| {DISPLAY_NAMES.get(cls, cls)} | {fmt(f'sensitivity[{cls}]')} | "
                 f"{fmt(f'specificity[{cls}]')} | {fmt(f'precision[{cls}]')} | {fmt(f'auroc[{cls}]', False)} |")
results_md = "\n".join(lines)
(OUT_DIR / "results_summary.md").write_text(results_md + "\n")

from IPython.display import Markdown, display
display(Markdown(results_md))
print("\nPer-seed results saved to", OUT_DIR / "test_metrics_per_seed.csv")

## 9. Figures: training curves, confusion matrix, reliability diagram

In [ ]:
# The deployed model is chosen on VALIDATION loss, never on test results.
best = min(runs, key=lambda r: r["val_loss"])
print(f"Selected seed {best['seed']} (lowest validation loss {best['val_loss']:.4f})")

# Training curves
h = best["history"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, key in zip(axes, ["loss", "accuracy"]):
    ax.plot(h[key], label="train")
    ax.plot(h[f"val_{key}"], label="validation")
    ax.axvline(h["stage1_epochs"] - 0.5, color="grey", ls="--", lw=1, label="start fine-tuning")
    ax.set_xlabel("epoch"); ax.set_title(key); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=120); plt.show()

# Confusion matrix (counts and row-normalised = per-class sensitivity on the diagonal)
names = [DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES]
cm = best["cm"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, mat, title, f in [(axes[0], cm, "Counts", "d"),
                          (axes[1], cm / cm.sum(axis=1, keepdims=True), "Row-normalised", ".2f")]:
    ax.imshow(mat, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, format(mat[i, j], f), ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() / 2 else "black")
    ax.set_xticks(range(NUM_CLASSES), names, rotation=20, ha="right", fontsize=8)
    ax.set_yticks(range(NUM_CLASSES), names, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Test confusion matrix: {title}")
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=120); plt.show()

# Reliability diagram before / after temperature scaling
def reliability(ax, y, probs, title, n_bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges = np.linspace(0, 1, n_bins + 1)
    centers, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            centers.append(conf[m].mean()); accs.append(correct[m].mean())
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
    ax.plot(centers, accs, "o-", label="model")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE = {expected_calibration_error(y, probs):.3f}"); ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
reliability(axes[0], y_test, best["p_test"], "Before temperature scaling")
reliability(axes[1], y_test, apply_temperature(best["p_test"], best["T"]), f"After (T = {best['T']:.2f})")
plt.tight_layout(); plt.savefig(FIG_DIR / "reliability_diagram.png", dpi=120); plt.show()

## 10. Save the selected model

In [ ]:
MODEL_PATH = OUT_DIR / "bcdc_mobilenetv2.keras"
best["model"].save(MODEL_PATH)
(OUT_DIR / "model_meta.json").write_text(json.dumps({
    "class_names": CLASS_NAMES,
    "display_names": DISPLAY_NAMES,
    "img_size": list(IMG_SIZE),
    "input": "RGB pixels in [0, 255]; preprocessing is inside the model",
    "temperature": best["T"],
    "seed": best["seed"],
    "min_confidence": MIN_CONFIDENCE,
}, indent=2))
print("Saved", MODEL_PATH, "and", OUT_DIR / "model_meta.json")

## 11. Inference on a single image

`predict_image` accepts a file path or an uploaded file and:

* converts **any** image mode (RGBA, grayscale, palette, CMYK) to RGB and respects EXIF rotation;
* resizes it the same way as in training;
* uses the model's softmax output **once**. The old code applied `tf.nn.softmax` a second time, which squashed every "confidence" into roughly 21–58%;
* applies the fitted temperature, so the reported probabilities are calibrated;
* reports **"uncertain"** below `MIN_CONFIDENCE`;
* warns when the image doesn't look like a grayscale scan, which catches colour photos.

> This colour check is a **heuristic, not a real out-of-distribution detector**: a grayscale photo of anything will pass it. Proper OOD rejection is on the roadmap. The model also has no "no tumour" class, so it will name a tumour type for *every* image, including healthy scans.

The old cost-estimate and "best hospitals" output has been **removed**. It printed hard-coded, unsourced figures next to a diagnosis.

In [ ]:
def load_rgb(src):
    im = Image.open(src)
    im = ImageOps.exif_transpose(im)
    return im.convert("RGB")


def looks_like_grayscale_scan(im_rgb, max_mean_chroma=8.0):
    a = np.asarray(im_rgb, dtype=np.float32)
    chroma = (np.abs(a[..., 0] - a[..., 1]) + np.abs(a[..., 1] - a[..., 2])).mean() / 2
    return chroma <= max_mean_chroma


def predict_image(src, model=None, temperature=None, show=True):
    model = best["model"] if model is None else model
    temperature = best["T"] if temperature is None else temperature
    im = load_rgb(src)
    x = tf.image.resize(np.asarray(im)[None], IMG_SIZE)        # same resize as training
    x = tf.round(x)                                             # training images were rounded to uint8 before caching
    probs = apply_temperature(model.predict(x, verbose=0), temperature)[0]   # single softmax + calibration
    order = np.argsort(probs)[::-1]
    top = order[0]

    if show:
        plt.figure(figsize=(3.2, 3.2)); plt.imshow(im); plt.axis("off"); plt.show()
    if not looks_like_grayscale_scan(im):
        print("⚠️  This image is in colour. It doesn't look like an MRI slice, so the prediction below is meaningless.")
    for k in order:
        print(f"  {DISPLAY_NAMES.get(CLASS_NAMES[k], CLASS_NAMES[k]):<34s} {100 * probs[k]:5.1f}%")
    if probs[top] < MIN_CONFIDENCE:
        print(f"→ Uncertain (top probability {100 * probs[top]:.1f}% < {100 * MIN_CONFIDENCE:.0f}%).")
    else:
        print(f"→ Most likely: {DISPLAY_NAMES.get(CLASS_NAMES[top], CLASS_NAMES[top])} ({100 * probs[top]:.1f}%)")
    print("This is a research prototype, not a diagnosis. Please discuss any MRI findings with a "
          "neurologist or neuro-oncologist.")
    return {CLASS_NAMES[k]: float(probs[k]) for k in range(NUM_CLASSES)}


# Example on one held-out test image
example = clean[clean["split"] == "test"].iloc[0]
print("True class:", DISPLAY_NAMES.get(example["cls"], example["cls"]))
_ = predict_image(example["path"])

In [ ]:
def run_app():
    # Upload one or more images in Colab and classify them.
    try:
        from google.colab import files
    except ImportError:
        print("The upload widget only works in Google Colab. Use predict_image('path/to/image.png') instead.")
        return
    for name, data in files.upload().items():
        print(f"\n=== {name} ===")
        predict_image(io.BytesIO(data))

# Uncomment to upload your own image:
# run_app()

## 12. Known limitations

* **No patient-level split.** The dataset has no patient IDs. Grouping near-duplicates removes copied images but not different slices of the same patient, so test scores may be optimistic.
* **Single dataset, single source.** All numbers come from one public dataset. Performance on scans from other hospitals or scanners is unknown until it is tested on an external dataset (see the roadmap in the README).
* **No healthy class.** The model always predicts a tumour type.
* **The `brain_tumor` class is underspecified** in the source dataset, so its predictions are hard to interpret clinically.
* **2D slices only.** The model sees one JPEG/PNG slice, not a full multi-sequence MRI study.